# Notebook 04 - baselines, tuning and the in-era check
Three models of increasing complexity, tuned without leakage, and checked on a normal (non-regime-change) season first. Reads `data/model_table.csv.gz`; writes the tuned parameters to `results/xgb_params.json` for notebooks 05-07.

1. **Physics baseline** - linear regression with a per-compound intercept and tyre-age slope plus a single fuel slope, following Heilmeier's lap-time decomposition (lap time = base + tyre wear + fuel mass). Fully interpretable.
2. **XGBoost** - gradient boosting on the full feature set, the community-standard model and what the F1 StratLab benchmark uses.
3. **Residual model** - XGBoost trained on the physics baseline's residuals. The bet: physical relationships (fuel burn, tyre wear) should transfer across the regulation change better than raw learned patterns.

**Rule we hold to:** never split laps randomly. Laps from one race are highly correlated (same car, same driver, same track, same day), so a random split leaks. Every split here is by race or by season.

In [1]:
import os, sys
# Run from the repo root so data/, figures/ and results/ resolve, and make src/ importable.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.path.abspath('src'))

import json
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV
from f1lap.features import encode_features
from f1lap.models import SEED, make_xgb, evaluate_split, PARAMS_PATH

np.random.seed(SEED)
os.makedirs('results', exist_ok=True)

df = pd.read_csv('data/model_table.csv.gz')
X_all = encode_features(df)          # one aligned matrix, sliced per split
y_all = df['lap_delta_s']
FEATURE_COLS = list(X_all.columns)

print('Model table:', df.shape, '| feature matrix:', X_all.shape)

Model table: (90488, 24) | feature matrix: (90488, 20)


### 1. Hyperparameter tuning
XGBoost is tuned with **GroupKFold grouped by race**, on the **training seasons only (2022-2025)**, so 2026 never influences the chosen parameters. Grouping by race keeps all of a race's laps in the same fold; a random K-fold would put laps from the same race in both train and validation and report an over-optimistic score.

In [2]:
train_era = df.season < 2026
groups = pd.factorize(df['season'].astype(str) + '_' + df['event'])[0]
grid = {'max_depth': [4, 6], 'n_estimators': [300, 600], 'learning_rate': [0.03, 0.08]}
search = GridSearchCV(make_xgb(n_jobs=1), grid, scoring='neg_mean_absolute_error',
                      cv=GroupKFold(n_splits=5), n_jobs=-1)
search.fit(X_all[train_era], y_all[train_era].values, groups=groups[train_era.to_numpy()])
XGB_PARAMS = dict(search.best_params_)
json.dump(XGB_PARAMS, open(PARAMS_PATH, 'w'), indent=1)
print('Chosen XGBoost params:', XGB_PARAMS)
print('Best grouped-CV MAE: %.4f s' % (-search.best_score_))

Chosen XGBoost params: {'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 300}
Best grouped-CV MAE: 0.5790 s


These parameters are used by the XGBoost and residual models in every later experiment.

### 2. In-era check (train 2022-2024, test 2025)
A within-distribution sanity check before we look at 2026: the error here is the bar we judge the regime shift against.

In [3]:
tr = df.season.isin([2022, 2023, 2024]); te = df.season == 2025
inera = evaluate_split(df, X_all, y_all, tr, te, XGB_PARAMS).round(3)
print('In-era (train 2022-2024, test 2025), MAE/RMSE in seconds:')
inera

In-era (train 2022-2024, test 2025), MAE/RMSE in seconds:


,MAE,RMSE
Physics,0.758,1.003
XGBoost,0.587,0.789
Residual,0.582,0.786


**Read:** XGBoost and the residual model land at **~0.59 s and ~0.58 s** MAE, with the physics baseline a clear floor at ~0.76 s. That's higher than the ~0.39 s F1 StratLab figure, but it's a loose reference rather than a like-for-like number: StratLab predicts raw lap time on 2023-2025, while we predict the race-median-relative *delta*, which already has the easy part (circuit length) removed. What matters is that the pipeline is sound and stable: the two boosted models roughly match each other and sit well below the physics floor. RMSE is about 35% above MAE, which tells us errors have a tail: most laps are predicted well, and a minority of laps are missed by a lot.